# Hệ thống AI đầy đủ — A/B trên dev (rule → +LLM vét → +semantic linking)
**Settings**: Accelerator = **GPU T4** · Internet = **On**. Save & Run All (~30-45').

Đo 3 cấu hình trên 60 file dev (metric BTC), thêm dần tầng AI:
- **A) rule + lexical** — floor (leaderboard 32.74)
- **B) hybrid + lexical** — rule + **LLM (Qwen 3B few-shot) vét bệnh/thuốc lạ**
- **C) hybrid + semantic** — thêm **bge-m3 linking** (ontological reasoning) khi lexical bí

LLM chỉ GỢI Ý cụm ≤6 từ (CHẨN_ĐOÁN/THUỐC), span snap về văn bản, không đè rule → floor an toàn.

In [ ]:
# 1) Code
%cd /kaggle/working
![ -d viettel-medreason ] || git clone https://github.com/tienndat1904/viettel-medreason.git
%cd /kaggle/working/viettel-medreason
!git pull -q

In [ ]:
# 2) Env — LLM (transformers/bnb/peft) + semantic (FlagEmbedding). KHÔNG động numpy.
!pip install -q -r requirements.txt
!pip install -q -U bitsandbytes peft accelerate
!pip install -q FlagEmbedding
import numpy, torch, transformers
print('numpy', numpy.__version__, '| torch', torch.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not numpy.__version__.startswith('2'):
    raise SystemExit('numpy != 2.x -> Restart & Run All')

In [ ]:
# 3) Build index semantic WHO (cho cấu hình C) + devset input
!python src/kb/build_icd_index.py --kb data/kb/icd10_vn.parquet --emb data/kb/icd_who_emb.npy --meta data/kb/icd_who_meta.parquet
import os, shutil, glob
os.makedirs('devset/input', exist_ok=True)
for g in glob.glob('data/dev/gold/*.json'):
    n = os.path.basename(g)[:-5]
    shutil.copy(f'data/test/input/{n}.txt', f'devset/input/{n}.txt')
print('devset:', len(glob.glob('devset/input/*.txt')), 'file')

In [ ]:
import yaml
def set_link(b):
    c = yaml.safe_load(open('configs/config.yaml', encoding='utf-8'))
    c['linking']['backend'] = b
    yaml.safe_dump(c, open('configs/config.yaml','w',encoding='utf-8'), allow_unicode=True, sort_keys=False)
    print('linking.backend =', b)

In [ ]:
# 5) A) RULE + lexical (floor)
set_link('lexical')
!python src/pipeline.py --input devset/input --output dev_A --backend rule
print('===== A) RULE + lexical =====')
!python src/eval/official_scorer.py --pred dev_A --gold data/dev/gold

In [ ]:
# 6) B) HYBRID (rule + LLM vét) + lexical — Qwen 3B few-shot tải ~6GB lần đầu
set_link('lexical')
!python src/pipeline.py --input devset/input --output dev_B --backend hybrid
print('===== B) HYBRID + lexical =====')
!python src/eval/official_scorer.py --pred dev_B --gold data/dev/gold

In [ ]:
# 7) C) HYBRID + SEMANTIC linking (Hệ AI đầy đủ)
set_link('semantic')
!python src/pipeline.py --input devset/input --output dev_C --backend hybrid
print('===== C) HYBRID + semantic =====')
!python src/eval/official_scorer.py --pred dev_C --gold data/dev/gold
set_link('lexical')

In [ ]:
# 8) LLM vét thêm bao nhiêu concept (A vs B) + link được mã
import json, glob, os
def load(d): return {os.path.basename(f): json.load(open(f, encoding='utf-8')) for f in glob.glob(d+'/*.json')}
A, B = load('dev_A'), load('dev_B')
add = 0; add_linked = 0
for fn in A:
    ak = {(c['text'], c['type']) for c in A[fn]}
    for c in B.get(fn, []):
        if (c['text'], c['type']) not in ak and c['type'] in ('CHẨN_ĐOÁN','THUỐC'):
            add += 1
            if c.get('candidates'):
                add_linked += 1
print('LLM vét thêm', add, 'concept CHẨN_ĐOÁN/THUỐC ngoài rule;', add_linked, 'link được mã.')
print('Chọn cấu hình FINAL cao nhất (A/B/C ở cell 5/6/7) để nộp.')